# Decision tree from scratch
Corrected, runnable version. Run the cells top to bottom.

**Two bugs were fixed from the original:**
1. In `gini_index`, the scoring lines were outside the `for group` loop — now correctly inside it.
2. In `split`, every `return` was un-indented (outside its `if`) — now correctly inside, so the tree actually grows.

In [1]:
def gini_index(groups, classes):
    n_instances = float(sum([len(group) for group in groups]))
    gini = 0.0
    for group in groups:
        size = float(len(group))
        if size == 0:
            continue
        score = 0.0
        labels = [row[-1] for row in group]
        for class_val in classes:
            p = labels.count(class_val) / size
            score += p * p
        gini += (1.0 - score) * (size / n_instances)
    return gini

In [2]:
def test_split(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[index] < value:
            left.append(row)
        else:
            right.append(row)
    return left, right

In [3]:
def get_best_split(dataset):
    class_values = list(set(row[-1] for row in dataset))
    best_index, best_value, best_score, best_groups = None, None, float('inf'), None
    for index in range(len(dataset[0]) - 1):
        for row in dataset:
            groups = test_split(index, row[index], dataset)
            gini = gini_index(groups, class_values)
            if gini < best_score:
                best_index, best_value, best_score, best_groups = index, row[index], gini, groups
    return {'index': best_index, 'value': best_value, 'groups': best_groups}

In [4]:
from collections import Counter

def to_terminal(group):
    outcomes = [row[-1] for row in group]
    return Counter(outcomes).most_common(1)[0][0]

In [5]:
def split(node, max_depth, min_size, depth):
    left, right = node['groups']
    del(node['groups'])
    # Check for no split
    if not left or not right:
        node['left'] = node['right'] = to_terminal(left + right)
        return
    # Check max depth
    if depth >= max_depth:
        node['left'], node['right'] = to_terminal(left), to_terminal(right)
        return
    # Process left child
    if len(left) <= min_size:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_best_split(left)
        split(node['left'], max_depth, min_size, depth + 1)
    # Process right child
    if len(right) <= min_size:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_best_split(right)
        split(node['right'], max_depth, min_size, depth + 1)

In [6]:
def build_tree(train, max_depth, min_size):
    root = get_best_split(train)
    split(root, max_depth, min_size, 1)
    return root

In [7]:
def predict(node, row):
    if row[node['index']] < node['value']:
        if isinstance(node['left'], dict):
            return predict(node['left'], row)
        else:
            return node['left']
    else:
        if isinstance(node['right'], dict):
            return predict(node['right'], row)
        else:
            return node['right']

## Use your dataset
Each row is `[feature1, feature2, ..., class]` — the **class label must be last**.

## The dataset in real life: a hiring decision

To make the numbers mean something, imagine each row is a **job candidate**, and we want to predict whether they get **hired**.

Each row is `[X1, X2, class]`:

| Column | Real-life meaning | Range |
|--------|-------------------|-------|
| `X1` | **Years of relevant experience** | 0 – 10 |
| `X2` | **Interview score** (panel rating) | 0 – 4 |
| `class` | **Outcome**: `0` = not hired, `1` = hired | 0 or 1 |

So a row like `[7.6, 2.8, 1]` means: *a candidate with 7.6 years of experience and an interview score of 2.8, who was hired.*

**What the tree discovers on its own:** after training, the decision boundary turns out to be

> **Hire if the interview score is strong (X2 ≥ 2.8), OR if experience is very high (X1 ≥ 8.9).**

This matches how real hiring often works: a great interview earns an offer regardless of experience, and a very experienced candidate can still get through on experience even after a middling interview. The tree wasn't *told* this rule — it found it by minimising Gini impurity, one split at a time.

**Why this dataset needs two questions.** No single line separates the hired from the not-hired: strong-interview candidates are hired whether junior or senior, so the tree must ask about the interview score *first*, then ask about experience only for the weaker-interview group. That two-question structure is exactly what you will see printed below when the tree is built.

In [8]:
# Your dataset: [X1, X2, class]
# Each row: [years_experience (X1), interview_score (X2), hired (class)]
dataset = [
    [2.8, 1.8, 0],
    [1.5, 2.3, 0],
    [3.4, 1.0, 0],
    [2.0, 3.6, 1],
    [3.1, 3.9, 1],
    [1.3, 3.3, 1],
    [7.6, 2.8, 1],
    [8.9, 1.2, 1],
    [7.4, 3.6, 1],
    [9.2, 0.9, 1],
]

# Build the tree (choose your own max_depth and min_size)
tree = build_tree(dataset, max_depth=3, min_size=1)

In [9]:
# Pretty-print the tree so you can see the questions it learned
def show(node, depth=0):
    if isinstance(node, dict):
        print('%sX%d < %.2f ?' % ('  ' * depth, node['index'] + 1, node['value']))
        show(node['left'], depth + 1)
        show(node['right'], depth + 1)
    else:
        print('%s=> class %s' % ('  ' * depth, node))

show(tree)

X2 < 2.80 ?
  X1 < 8.90 ?
    X1 < 2.80 ?
      => class 0
      => class 0
    X1 < 8.90 ?
      => class 1
      => class 1
  X1 < 2.00 ?
    => class 1
    X1 < 2.00 ?
      => class 1
      => class 1


In [10]:
# Predict every row and check against the true label
for row in dataset:
    prediction = predict(tree, row)
    print("features=%s  expected=%s  predicted=%s" % (row[:-1], row[-1], prediction))

features=[2.8, 1.8]  expected=0  predicted=0
features=[1.5, 2.3]  expected=0  predicted=0
features=[3.4, 1.0]  expected=0  predicted=0
features=[2.0, 3.6]  expected=1  predicted=1
features=[3.1, 3.9]  expected=1  predicted=1
features=[1.3, 3.3]  expected=1  predicted=1
features=[7.6, 2.8]  expected=1  predicted=1
features=[8.9, 1.2]  expected=1  predicted=1
features=[7.4, 3.6]  expected=1  predicted=1
features=[9.2, 0.9]  expected=1  predicted=1


In [11]:
# Predict a brand-new point (last value can be None - it is ignored)
new_point = [7.5, 1.5, None]
print("Prediction for", new_point[:-1], "->", predict(tree, new_point))

Prediction for [7.5, 1.5] -> 0
